In [2]:
# Cell 1: Basic counters and Cartesian product
import statecounter as sc

with sc.Manager():
    A = sc.Counter(2).named('A')
    B = sc.Counter(3).named('B')
    C = sc.product([A, B]).named('C')  # Cartesian product: 2 * 3 = 6 states
    
C.print_tree()
C.get_iteration_df().T

C [product, n=6]
├── A [leaf, n=2]
└── B [leaf, n=3]


C,0,1,2,3,4,5
A,0,1,0,1,0,1
B,0,0,1,1,2,2


In [3]:
# Cell 2: Stack - inactive branches show None
with sc.Manager():
    A = sc.Counter(2).named('A')
    B = sc.Counter(3).named('B')
    C = sc.stack([A,B]).named('C')  # Disjoint union: 2 + 3 = 5 states
    
C.print_tree()
C.get_iteration_df().T

C [stack, n=5]
├── A [leaf, n=2]
└── B [leaf, n=3]


C,0,1,2,3,4
A,0,1,None,None,None
B,None,None,0,1,2


In [4]:
# Cell 3: Slicing and repeat operations
with sc.Manager():
    A = sc.Counter(6).named('A')
    B = A[1:4].named('B')    # Slice: states 1,2,3 -> 3 states
    C = A[::2].named('C')    # Every other: 0,2,4 -> 3 states
    D = A[::-1].named('D')   # Reverse order: 5,4,3,2,1,0
    E = sc.repeat(A, 2).named('E')   # Repeat twice: 12 states

for counter in [B,C,D,E]:
    counter.print_tree()
    display(counter.get_iteration_df().T)

B [slice, n=3]
└── A [leaf, n=6]


B,0,1,2
A,1,2,3


C [slice, n=3]
└── A [leaf, n=6]


C,0,1,2
A,0,2,4


D [slice, n=6]
└── A [leaf, n=6]


D,0,1,2,3,4,5
A,5,4,3,2,1,0


E [repeat, n=12]
└── A [leaf, n=6]


E,0,1,2,3,4,5,6,7,8,9,10,11
A,0,1,2,3,4,5,0,1,2,3,4,5


In [5]:
# Cell 4: Sync operation - keeps counters with same num_states in lockstep
with sc.Manager():
    A = sc.Counter(4).named('A')
    B = sc.Counter(4).named('B')
    C = sc.sync(A, B).named('C')  # Both A and B track together
    
C.print_tree()
C.get_iteration_df().T

C [sync, n=4]
├── A [leaf, n=4]
└── B [leaf, n=4]


C,0,1,2,3
A,0,1,2,3
B,0,1,2,3


In [6]:
# Cell 5: Shuffle and interleave operations
with sc.Manager():
    A = sc.Counter(4).named('A')
    B = sc.shuffle(A, seed=42).named('B')  # Randomized order (deterministic)
    
    X = sc.Counter(3).named('X')
    Y = sc.Counter(3).named('Y')
    Z = sc.interleave(X, Y).named('Z')  # Alternates: X0,Y0,X1,Y1,X2,Y2

Z.print_tree()
Z.get_iteration_df().T

Z [interleave, n=6]
├── X [leaf, n=3]
└── Y [leaf, n=3]


Z,0,1,2,3,4,5
X,0,None,1,None,2,None
Y,None,0,None,1,None,2


In [8]:
# Cell 6: Split operation - divide counter into parts
with sc.Manager() as mgr:
    A = sc.Counter(12).named('A')
    B, C, D = sc.split(A, 3, names=['B', 'C', 'D'])  # Equal: 4, 4, 4
    
    for ctr in [B,C,D]:
        ctr.print_tree()
        display(ctr.get_iteration_df().T)
    
    X = sc.Counter(10).named('X')
    Y, Z = sc.split(X, [1, 4], names=['Y', 'Z'])  # Proportional: 2, 8

    for ctr in [Y,Z]:
        ctr.print_tree()
        display(ctr.get_iteration_df().T)

B [slice, n=4]
└── A [leaf, n=12]


B,0,1,2,3
A,0,1,2,3


C [slice, n=4]
└── A [leaf, n=12]


C,0,1,2,3
A,4,5,6,7


D [slice, n=4]
└── A [leaf, n=12]


D,0,1,2,3
A,8,9,10,11


Y [slice, n=2]
└── X [leaf, n=10]


Y,0,1
X,0,1


Z [slice, n=8]
└── X [leaf, n=10]


Z,0,1,2,3,4,5,6,7
X,2,3,4,5,6,7,8,9


In [11]:

# Cell 7: State propagation through composite counters
with sc.Manager():
    A = sc.Counter(2).named('A')
    B = sc.Counter(3).named('B')
    C = sc.product([A, B]).named('C')
    D = sc.Counter(2).named('D')
    E = sc.stack([C,D]).named('E')  # product([A, B]) + D = 6 + 2 = 8 states
    
    E.print_tree()
    display(E.get_iteration_df().T)
    
    # Setting E.state propagates values through the tree
    E.state = 4  
    for ctr in [E,A,B,C,D]:
        print(f'{ctr.name}.state={ctr.state}')


E [stack, n=8]
├── C [product, n=6]
│   ├── A [leaf, n=2]
│   └── B [leaf, n=3]
└── D [leaf, n=2]


E,0,1,2,3,4,5,6,7
C,0,1,2,3,4,5,None,None
A,0,1,0,1,0,1,None,None
B,0,0,1,1,2,2,None,None
D,None,None,None,None,None,None,0,1


E.state=4
A.state=0
B.state=2
C.state=4
D.state=None


In [ ]:
# Cell 8: Iteration and state management
with sc.Manager():
    A = sc.Counter(3).named('A')
    B = sc.Counter(2).named('B')
    C = sc.product([A, B]).named('C')
    
    # Iterate through all states
    states = [(A.state, B.state) for _ in C]
    
    # Manual state control
    C.reset()      # Set to state 0
    C.advance()    # Move to next state
    C.state = 4    # Set directly

C.print_tree()
C.get_iteration_df()